# Offline mutation benchmark

Run independent, seeded EVCC sessions against the repository's in-process SECC mock. Each trial dispatches exactly one selected CurrentDemand mutation and records what the mock actually returned. A rejection is an expected conformance observation, not a vulnerability or a real-hardware result. No Java, CAN, raw Ethernet, or LLM completion is used.

In [ ]:
import json
import os
import random
import subprocess
import sys
from pathlib import Path

# Hosted Colab starts in /content; reuse a checkout if one is already present.
in_colab = 'google.colab' in sys.modules
if in_colab:
    project_dir = Path('/content/evcc-sift-mutator')
    if not project_dir.exists():
        subprocess.run([
            'git', 'clone', '--depth', '1',
            'https://github.com/JavaStudentAlex/evcc-sift-mutator.git',
            str(project_dir),
        ], check=True)
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-e', f'{project_dir}[dev]'
    ], cwd=project_dir, check=True)
else:
    project_dir = Path.cwd()
if not (project_dir / 'simulator' / 'secc_mock.py').is_file():
    raise RuntimeError('Start a local notebook from the repository root, or use hosted Colab.')
sys.path.insert(0, str(project_dir))
print('Fixture:', project_dir, '(hosted Colab)' if in_colab else '(local)')

In [ ]:
# Component tests run entirely offline, without a charger or model proxy.
subprocess.run([
    sys.executable, '-m', 'pytest', '-q',
    'tests/test_simulator.py',
    'tests/test_mutation_engine.py',
    'tests/test_self_healing.py',
], cwd=project_dir, check=True)

In [ ]:
from mutation.operators import mutate_numerical_boundary, mutate_semantic_spoof
from mutation.self_healing import SelfHealingEngine
from simulator.hooks import GLOBAL_HOOKS
from simulator.secc_mock import SeccMockServer
from simulator.state_machine import Iso15118EvccStateMachine

GLOBAL_HOOKS.clear()
control_server = SeccMockServer()
control_machine = Iso15118EvccStateMachine(transport_sender=control_server.process_message)
try:
    control_complete = control_machine.run_full_session(current_demand_cycles=1)
finally:
    GLOBAL_HOOKS.clear()
control = {
    'complete': control_complete,
    'message_count': len(control_server.message_history),
    'observations': list(control_server.anomalies_detected),
}
assert control_complete, 'Canonical mock session must pass before comparing mutations.'
print('Canonical control:', json.dumps(control, indent=2))

In [ ]:
def run_trial(operator, seed):
    rng_state = random.getstate()
    random.seed(seed)
    server = SeccMockServer()
    healing = SelfHealingEngine()
    selected = {'base': None, 'payload': None, 'candidate': None, 'response': None}

    def mutate_once(request, state):
        if state != 'CURRENT_DEMAND' or selected['payload'] is not None:
            return request
        if not healing.validate_payload(request).is_valid:
            raise ValueError('Canonical CurrentDemand request failed validation')
        candidate, _ = healing.execute_with_self_healing(
            request, lambda base, error: operator(base)
        )
        selected.update(base=request, payload=candidate.mutated_payload, candidate=candidate)
        return selected['payload']

    def send(request):
        response = server.process_message(request)
        if request is selected['payload']:
            selected['response'] = response
        return response

    GLOBAL_HOOKS.clear()
    GLOBAL_HOOKS.register_pre_send(mutate_once)
    machine = Iso15118EvccStateMachine(transport_sender=send)
    try:
        complete = machine.run_full_session(current_demand_cycles=1)
    finally:
        GLOBAL_HOOKS.clear()
        random.setstate(rng_state)
    candidate = selected['candidate']
    if candidate is None:
        raise RuntimeError('No mutation reached CurrentDemand; inspect nominal control')
    payload = selected['payload']
    dispatches = sum(item is payload for item in server.message_history)
    assert dispatches == 1, 'A benchmark case must dispatch its mutation once'
    response = selected['response']
    response_code = getattr(response, 'response_code', None)
    response_code = getattr(response_code, 'value', response_code)
    return {
        'seed': seed,
        'operator': operator.__name__,
        'arm': candidate.arm,
        'description': candidate.description,
        'base_request': {'name': type(selected['base']).__name__, 'fields': selected['base'].model_dump(mode='json', exclude={'session_id'})},
        'sent_request': {'name': type(payload).__name__, 'fields': payload.model_dump(mode='json', exclude={'session_id'})},
        'received_response': {'name': type(response).__name__ if response else None, 'code': response_code},
        'outcome': 'no_response' if response is None else 'accepted' if response_code == 'OK' else 'rejected',
        'session_complete': complete,
        'dispatches': dispatches,
        'validation_passed': healing.validate_payload(payload).is_valid,
        'failed_validation_attempts': len(healing.repair_history),
        'fixture_observations': list(server.anomalies_detected),
    }

seed = 2026
operators = [mutate_numerical_boundary, mutate_semantic_spoof]
trials = [run_trial(operators[index % len(operators)], seed + index) for index in range(6)]
counts = {outcome: sum(case['outcome'] == outcome for case in trials) for outcome in ('accepted', 'rejected', 'no_response')}
benchmark = {'fixture': 'in-process SeccMockServer (JSON simulation, not DIN EXI)', 'seed': seed, 'control': control, 'counts': counts, 'trials': trials}
print('Outcomes:', json.dumps(counts, indent=2))
print('Measured trial data:', json.dumps(trials, indent=2))

In [ ]:
# Optional export: set SIM_BENCHMARK_OUTPUT to a path before running this cell.
output = os.environ.get('SIM_BENCHMARK_OUTPUT')
if output:
    destination = Path(output).expanduser()
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(json.dumps(benchmark, indent=2) + '\n', encoding='utf-8')
    print('Wrote benchmark:', destination)
else:
    print('No artifact written; set SIM_BENCHMARK_OUTPUT to export the measured JSON.')

## Optional model-gateway check

CLIProxyAPI is not a charger simulator. The check below makes **only** a GET request to its `/v1/models` endpoint if `CLIPROXYAPI_BASE_URL` is set. It never submits a test vector or a model completion. For a local runtime, an example value is `http://127.0.0.1:8317`; hosted Colab cannot normally reach a service on your laptop's localhost. An API key can be supplied in `CLIPROXYAPI_API_KEY`; its value is never printed.

In [ ]:
import urllib.request

proxy_base = os.environ.get('CLIPROXYAPI_BASE_URL', '').rstrip('/')
if proxy_base:
    key = os.environ.get('CLIPROXYAPI_API_KEY')
    headers = {'Authorization': f'Bearer {key}'} if key else {}
    request = urllib.request.Request(f'{proxy_base}/v1/models', headers=headers, method='GET')
    try:
        with urllib.request.urlopen(request, timeout=3) as reply:
            models = json.load(reply)
            print('Gateway status:', reply.status, '| available models:', len(models.get('data', [])))
    except Exception as error:
        print('Gateway check unavailable:', type(error).__name__)
else:
    print('Gateway check skipped (no CLIPROXYAPI_BASE_URL).')